# UNO1D-H10 on 1D Burgers: official temporal protocol

This notebook gives UNO the same temporal information used by the official
autoregressive Navier--Stokes experiment:

\[
(u_{n-9},\ldots,u_n)\longmapsto \widehat u_{n+1}.
\]

The predicted state is appended to the ten-state history and the oldest state
is removed. The network still outputs only one state per call; it is therefore
**not a windowed-output model**.

Official UNO settings retained here:

- input history: 10 exact states;
- output: one next state;
- autoregressive training and inference;
- 40 recursively predicted training steps;
- width 32 and factor 3/4;
- batch size 16;
- 500 epochs;
- Adam, learning rate 1e-3, weight decay 1e-5;
- StepLR every 100 epochs with factor 0.5.

The official repository has no 1D Burgers experiment. Consequently, the
spatial architecture is the official 1D UNO operator assembled in the same
seven-block U shape used by the temporal UNO, and the 201-snapshot trajectory
is divided into 10-input/40-output training segments. This should be reported
as `UNO-H10 (official temporal protocol, 1D adaptation)`, not as an exact
published Burgers reproduction.

**Fairness warning:** UNO-H10 receives the first ten exact test states,
whereas AWFNO receives only \(u_0\). Its first ten entries in the reported
full trajectory are therefore exact by construction.

Primary sources:

- https://arxiv.org/abs/2204.11127
- https://github.com/ashiq24/UNO/blob/main/ns_uno2d_main.py
- https://github.com/ashiq24/UNO/blob/main/ns_train_2d.py
- https://github.com/ashiq24/UNO/blob/main/integral_operators.py

In [ ]:
from google.colab import drive
drive.mount("/content/gdrive")

## Configuration

The optimizer uses a constant learning rate for every sample size. The
schedule, batch size, epoch budget, spatial/temporal resolution, split, and
checkpoint-selection rule are fixed across the sweep.

In [ ]:
from pathlib import Path

CONFIG = {
    "DATA_PATH": "/content/gdrive/MyDrive/anus/1D_Burgers_Sols_Nu0.001.hdf5",
    "OUTPUT_ROOT": "/content/gdrive/MyDrive/anus/burgers_uno1d_h10_official_protocol",

    # Exact AWFNO split and resolution
    "TRAIN_POOL_SIZE": 800,
    "N_VAL": 50,
    "N_TEST": 100,
    "TARGET_NX": 512,
    "TIME_SUBSAMPLE": 1,
    "SAMPLE_SIZES": [25, 50, 100, 200, 400, 800],
    "RUN_SAMPLE_SIZES": [25, 50, 100, 200, 400, 800],

    # Official temporal protocol
    "HISTORY_LENGTH": 10,
    "TRAIN_ROLLOUT_STEPS": 40,
    "TRAIN_WINDOW_STRIDE": 40,

    # Paper-style UNO1D architecture
    "WIDTH": 32,
    "FACTOR": 0.75,
    "MODES": [64, 48, 24, 24, 24, 48, 64],
    "NORMALIZE_BLOCKS": False,

    # Official UNO optimization settings
    "EPOCHS": 500,
    "BATCH_SIZE": 16,
    "EVAL_BATCH_SIZE": 16,
    "LR": 1.0e-3,
    "WEIGHT_DECAY": 1.0e-5,
    "SCHEDULER_STEP": 100,
    "SCHEDULER_GAMMA": 0.5,
    "VAL_ROLLOUT_INTERVAL": 10,

    # Reproducibility and execution
    "BASE_SEED": 141234,
    "FORCE_RETRAIN": False,
    "RUN_EXACT_HASH_OVERLAP_CHECK": True,
    "INFERENCE_REPEATS": 3,
    "NUM_WORKERS": 0,
    "PIN_MEMORY": True,

    # Physical L2 rule used by the AWFNO benchmark
    "USE_TRAPEZOIDAL_TIME_L2": True,
    "INCLUDE_T0_IN_FULL_L2": True,

    # Outputs
    "SAVE_ROLLOUTS": False,
    "SAVE_PLOTS": True,
}

DATA_PATH = Path(CONFIG["DATA_PATH"])
OUTPUT_ROOT = Path(CONFIG["OUTPUT_ROOT"])
MODEL_DIR = OUTPUT_ROOT / "models"
RESULT_DIR = OUTPUT_ROOT / "results"
PLOT_DIR = OUTPUT_ROOT / "plots"
for directory in (OUTPUT_ROOT, MODEL_DIR, RESULT_DIR, PLOT_DIR):
    directory.mkdir(parents=True, exist_ok=True)

assert CONFIG["HISTORY_LENGTH"] == 10
assert CONFIG["TRAIN_ROLLOUT_STEPS"] == 40
assert CONFIG["WIDTH"] == 32
assert CONFIG["FACTOR"] == 0.75
assert CONFIG["BATCH_SIZE"] == 16
assert CONFIG["EPOCHS"] == 500

In [ ]:
import copy
import gc
import hashlib
import json
import math
import os
import random
import time
import warnings
from dataclasses import dataclass

import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils import weight_norm
from torch.utils.data import DataLoader, Dataset

warnings.filterwarnings(
    "ignore",
    message="`torch.nn.utils.weight_norm` is deprecated",
)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def json_ready(value):
    if isinstance(value, dict):
        return {str(k): json_ready(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_ready(v) for v in value]
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, (np.floating, np.integer)):
        return value.item()
    return value


with open(OUTPUT_ROOT / "config.json", "w") as stream:
    json.dump(json_ready(CONFIG), stream, indent=2)

## Load the exact dataset and split

In [ ]:
def choose_hdf5_keys(handle):
    datasets = {}

    def visitor(name, obj):
        if isinstance(obj, h5py.Dataset):
            datasets[name] = tuple(obj.shape)

    handle.visititems(visitor)
    print("HDF5 datasets")
    for name, shape in datasets.items():
        print(f"  {name}: {shape}")

    solution_candidates = [
        name for name, shape in datasets.items()
        if len(shape) == 3 and min(shape) > 1
    ]
    if not solution_candidates:
        raise RuntimeError("No three-dimensional solution dataset was found.")
    solution_key = max(solution_candidates, key=lambda k: np.prod(datasets[k]))

    one_dimensional = [
        name for name, shape in datasets.items() if len(shape) == 1
    ]
    x_key = next(
        (name for name in one_dimensional if "x-coordinate" in name.lower()),
        next((name for name in one_dimensional if name.lower().endswith("x")), None),
    )
    t_key = next(
        (name for name in one_dimensional if "t-coordinate" in name.lower()),
        next((name for name in one_dimensional if name.lower().endswith("t")), None),
    )
    if x_key is None:
        raise RuntimeError("Could not identify the spatial coordinate dataset.")
    return solution_key, x_key, t_key


def orient_solution(raw, x):
    """Return the solution in (trajectory, space, time) order."""
    if raw.ndim != 3:
        raise ValueError(f"Expected a rank-three solution tensor, got {raw.shape}.")
    nx = len(x)
    if raw.shape[2] == nx:
        # PDEBench Burgers layout: (trajectory, time, space)
        return np.transpose(raw, (0, 2, 1))
    if raw.shape[1] == nx:
        return raw
    raise ValueError(
        f"Could not identify the spatial axis: solution={raw.shape}, len(x)={nx}."
    )


required_rows = (
    CONFIG["TRAIN_POOL_SIZE"] + CONFIG["N_VAL"] + CONFIG["N_TEST"]
)
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at {DATA_PATH}. Edit CONFIG['DATA_PATH'] if needed."
    )

with h5py.File(DATA_PATH, "r") as handle:
    solution_key, x_key, t_key = choose_hdf5_keys(handle)
    raw_shape = handle[solution_key].shape
    if raw_shape[0] < required_rows:
        raise ValueError(
            f"Need at least {required_rows} trajectories, found {raw_shape[0]}."
        )
    raw = np.asarray(handle[solution_key][:required_rows], dtype=np.float32)
    x_original = np.asarray(handle[x_key][:], dtype=np.float32)
    t_original = (
        np.asarray(handle[t_key][:], dtype=np.float32)
        if t_key is not None else None
    )

U = orient_solution(raw, x_original)
del raw
gc.collect()

if CONFIG["TIME_SUBSAMPLE"] > 1:
    U = U[:, :, :: CONFIG["TIME_SUBSAMPLE"]]

original_nx = U.shape[1]
target_nx = CONFIG["TARGET_NX"]
if original_nx % target_nx != 0:
    raise ValueError(
        f"Spatial size {original_nx} is not divisible by TARGET_NX={target_nx}."
    )
spatial_stride = original_nx // target_nx
U = np.ascontiguousarray(U[:, ::spatial_stride, :], dtype=np.float32)
x_grid = np.ascontiguousarray(x_original[::spatial_stride], dtype=np.float32)

N_TOTAL, NX, NT = U.shape
if t_original is not None and len(t_original) == NT:
    t_grid = t_original.copy()
    DT_SNAPSHOT = float(np.mean(np.diff(t_grid)))
else:
    print(
        f"Warning: time-coordinate length "
        f"{None if t_original is None else len(t_original)} != Nt={NT}; "
        "using snapshot index, exactly as in the AWFNO benchmark."
    )
    t_grid = np.arange(NT, dtype=np.float64)
    DT_SNAPSHOT = 1.0

DX = float(np.mean(np.diff(x_grid)))

i0 = 0
i1 = CONFIG["TRAIN_POOL_SIZE"]
i2 = i1 + CONFIG["N_VAL"]
i3 = i2 + CONFIG["N_TEST"]
U_train_pool = U[i0:i1]
U_val = U[i1:i2]
U_test = U[i2:i3]

print("\nLoaded geometry and split")
print("-------------------------")
print("U shape          :", U.shape)
print("U_train_pool     :", U_train_pool.shape, "ids 0:800")
print("U_val            :", U_val.shape, "ids 800:850")
print("U_test           :", U_test.shape, "ids 850:950")
print("NX, NT           :", NX, NT)
print("DX, DT_SNAPSHOT  :", DX, DT_SNAPSHOT)

assert U_train_pool.shape == (800, 512, 201)
assert U_val.shape == (50, 512, 201)
assert U_test.shape == (100, 512, 201)


def trajectory_hashes(array):
    hashes = []
    for trajectory in array:
        contiguous = np.ascontiguousarray(trajectory)
        hashes.append(hashlib.sha256(contiguous.view(np.uint8)).hexdigest())
    return set(hashes)


if CONFIG["RUN_EXACT_HASH_OVERLAP_CHECK"]:
    train_hash = trajectory_hashes(U_train_pool)
    val_hash = trajectory_hashes(U_val)
    test_hash = trajectory_hashes(U_test)
    print("\nExact trajectory overlap")
    print("------------------------")
    print("train_pool vs val :", len(train_hash & val_hash))
    print("train_pool vs test:", len(train_hash & test_hash))
    print("val vs test       :", len(val_hash & test_hash))
    assert not (train_hash & val_hash)
    assert not (train_hash & test_hash)
    assert not (val_hash & test_hash)
    del train_hash, val_hash, test_hash

## Physical \(L^2\) metrics

In [ ]:
def time_weights(nt):
    weights = np.ones(nt, dtype=np.float64)
    if CONFIG["USE_TRAPEZOIDAL_TIME_L2"] and nt > 1:
        weights[0] = 0.5
        weights[-1] = 0.5
    return weights


def l2_per_timestep_np(prediction, truth):
    if prediction.shape != truth.shape:
        raise ValueError(f"Shape mismatch: {prediction.shape} vs {truth.shape}")
    error = prediction.astype(np.float64) - truth.astype(np.float64)
    return np.sqrt(np.sum(error * error, axis=1) * DX + 1.0e-30)


def rollout_metrics(prediction, truth):
    l2_t = l2_per_timestep_np(prediction, truth)
    if CONFIG["INCLUDE_T0_IN_FULL_L2"]:
        used = l2_t
    else:
        used = l2_t[:, 1:]
    weights = time_weights(used.shape[1])[None, :]
    full = np.sqrt(
        np.sum(weights * used * used, axis=1) * DT_SNAPSHOT + 1.0e-30
    )
    final = l2_t[:, -1]
    return {
        "full_mean": float(np.mean(full)),
        "full_std": float(np.std(full)),
        "full_median": float(np.median(full)),
        "final_mean": float(np.mean(final)),
        "final_std": float(np.std(final)),
        "final_median": float(np.median(final)),
        "mean_t": np.mean(l2_t, axis=0),
        "median_t": np.median(l2_t, axis=0),
        "std_t": np.std(l2_t, axis=0),
        "all_full": full,
        "all_final": final,
    }


def physical_field_l2_torch(prediction, truth):
    return torch.sqrt(
        torch.sum((prediction - truth) ** 2, dim=1) * DX + 1.0e-30
    )

## Official-style 10-input/40-output training segments

The published temporal experiment uses the first 10 states as input and
recursively predicts 40 future states. Our trajectories contain 201 states,
so non-overlapping 40-step forecast segments are used to expose the model to
the full training interval. Exact states initialize each training segment;
during the following 40 steps, only the model's predictions are fed back.

In [ ]:
class RolloutWindowDataset(Dataset):
    def __init__(self, trajectories, history_length, forecast_length, stride):
        self.U = np.asarray(trajectories, dtype=np.float32)
        self.history_length = int(history_length)
        self.forecast_length = int(forecast_length)
        maximum_start = (
            self.U.shape[2] - self.history_length - self.forecast_length
        )
        if maximum_start < 0:
            raise ValueError(self.U.shape)
        starts = list(range(0, maximum_start + 1, int(stride)))
        if starts[-1] != maximum_start:
            starts.append(maximum_start)
        self.starts = starts

    def __len__(self):
        return self.U.shape[0] * len(self.starts)

    def __getitem__(self, index):
        trajectory = index // len(self.starts)
        start = self.starts[index % len(self.starts)]
        middle = start + self.history_length
        end = middle + self.forecast_length
        history = torch.from_numpy(self.U[trajectory, :, start:middle])
        future = torch.from_numpy(self.U[trajectory, :, middle:end])
        return history, future


def relative_l2_sum(prediction, truth):
    batch = prediction.shape[0]
    difference = torch.linalg.vector_norm(
        (prediction - truth).reshape(batch, -1), dim=1
    )
    reference = torch.linalg.vector_norm(
        truth.reshape(batch, -1), dim=1
    )
    return torch.sum(difference / (reference + 1.0e-12))

## Paper-style one-dimensional UNO

The official UNO block computes

\[
v_{l+1}=\sigma\!\left(\mathcal{K}_{l}v_l+
W_lv_l\right),
\]

where \(\mathcal{K}_l\) is a truncated Fourier operator and \(W_l\) is
a pointwise \(1\times1\) convolution. Both paths are resampled to the target
resolution of the block. The spatial resolutions form a U and features from
matching encoder and decoder resolutions are concatenated.

We keep the official seven-block layout and width/factor configuration. The
original code uses 22/14/6 retained modes on a 64-point axis; here we use
64/48/24 modes across the 512/384/256/128 grids so the adaptation is not
artificially mode-starved. The output remains one next state.

In [ ]:
class SpectralConv1dUNO(nn.Module):
    def __init__(self, in_channels, out_channels, modes):
        super().__init__()
        self.in_channels = int(in_channels)
        self.out_channels = int(out_channels)
        self.modes = int(modes)
        scale = (1.0 / (2.0 * self.in_channels)) ** 0.5
        self.weight = nn.Parameter(
            scale
            * torch.randn(
                self.in_channels,
                self.out_channels,
                self.modes,
                dtype=torch.cfloat,
            )
        )

    def forward(self, x, output_length):
        if x.ndim != 3:
            raise ValueError(f"Expected (batch, channels, x), received {x.shape}.")
        output_length = int(output_length)
        x_ft = torch.fft.rfft(x.float(), dim=-1, norm="forward")
        retained = min(
            self.modes,
            x_ft.shape[-1],
            output_length // 2 + 1,
        )
        out_ft = torch.zeros(
            x.shape[0],
            self.out_channels,
            output_length // 2 + 1,
            dtype=torch.cfloat,
            device=x.device,
        )
        out_ft[:, :, :retained] = torch.einsum(
            "bix,iox->box",
            x_ft[:, :, :retained],
            self.weight[:, :, :retained],
        )
        return torch.fft.irfft(
            out_ft, n=output_length, dim=-1, norm="forward"
        )


class PointwiseOp1dUNO(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Conv1d(int(in_channels), int(out_channels), 1)

    def forward(self, x, output_length):
        x = self.conv(x)
        if x.shape[-1] != int(output_length):
            # This matches the official linear pointwise resampling. The
            # repository's antialias=True option is omitted because current
            # PyTorch only supports antialiasing for 4D bilinear/bicubic input.
            x = F.interpolate(
                x,
                size=int(output_length),
                mode="linear",
                align_corners=True,
            )
        return x


class OperatorBlock1dUNO(nn.Module):
    def __init__(
        self,
        in_channels,
        out_channels,
        modes,
        normalize=False,
        nonlinearity=True,
    ):
        super().__init__()
        self.spectral = SpectralConv1dUNO(
            in_channels, out_channels, modes
        )
        self.pointwise = PointwiseOp1dUNO(in_channels, out_channels)
        self.normalize = bool(normalize)
        self.nonlinearity = bool(nonlinearity)
        if self.normalize:
            self.norm = nn.InstanceNorm1d(int(out_channels), affine=True)

    def forward(self, x, output_length):
        x = (
            self.spectral(x, output_length)
            + self.pointwise(x, output_length)
        )
        if self.normalize:
            x = self.norm(x)
        if self.nonlinearity:
            x = F.gelu(x)
        return x


class UNO1D_H10(nn.Module):
    """Ten-state-history UNO with a one-state autoregressive output."""

    def __init__(self, x_grid, config):
        super().__init__()
        width = int(config["WIDTH"])
        factor = float(config["FACTOR"])
        modes = list(config["MODES"])
        normalize = bool(config["NORMALIZE_BLOCKS"])

        c0 = int(2 * factor * width)
        c1 = int(4 * factor * width)
        c2 = int(8 * factor * width)
        if (c0, c1, c2) != (48, 96, 192):
            raise ValueError((c0, c1, c2))

        # Input channels: ten states, sin(2*pi*x), cos(2*pi*x).
        x_tensor = torch.tensor(x_grid, dtype=torch.float32)
        x_normalized = (
            (x_tensor - x_tensor.min())
            / (x_tensor.max() - x_tensor.min() + 1.0e-12)
        )
        coordinates = torch.stack(
            [
                torch.sin(2.0 * math.pi * x_normalized),
                torch.cos(2.0 * math.pi * x_normalized),
            ],
            dim=-1,
        )
        self.register_buffer(
            "coordinates", coordinates.unsqueeze(0), persistent=False
        )

        self.fc = nn.Linear(config["HISTORY_LENGTH"] + 2, width // 2)
        self.fc0 = nn.Linear(width // 2, width)

        self.L0 = OperatorBlock1dUNO(
            width, c0, modes[0], normalize=normalize
        )
        self.L1 = OperatorBlock1dUNO(
            c0, c1, modes[1], normalize=normalize
        )
        self.L2 = OperatorBlock1dUNO(
            c1, c2, modes[2], normalize=normalize
        )
        self.L3 = OperatorBlock1dUNO(
            c2, c2, modes[3], normalize=normalize
        )
        self.L4 = OperatorBlock1dUNO(
            c2, c1, modes[4], normalize=normalize
        )
        self.L5 = OperatorBlock1dUNO(
            2 * c1, c0, modes[5], normalize=normalize
        )
        self.L6 = OperatorBlock1dUNO(
            2 * c0, width, modes[6], normalize=normalize
        )

        self.fc1 = nn.Linear(2 * width, 4 * width)
        self.fc2 = nn.Linear(4 * width, 1)

    def forward(self, history):
        if history.ndim != 3:
            raise ValueError(
                f"Expected (batch, space, history), received {history.shape}."
            )
        batch, nx, history_length = history.shape
        if history_length != CONFIG["HISTORY_LENGTH"]:
            raise ValueError(
                f"Expected history {CONFIG['HISTORY_LENGTH']}, got {history_length}."
            )
        coordinates = self.coordinates.expand(batch, -1, -1)
        x = torch.cat([history, coordinates], dim=-1)
        x_fc = F.gelu(self.fc(x))
        x_fc0 = F.gelu(self.fc0(x_fc)).permute(0, 2, 1).contiguous()

        r0 = int(nx * CONFIG["FACTOR"])
        r1 = nx // 2
        r2 = nx // 4
        if (r0, r1, r2) != (384, 256, 128):
            raise ValueError(f"Expected 512-point input, obtained nx={nx}.")

        x_c0 = self.L0(x_fc0, r0)
        x_c1 = self.L1(x_c0, r1)
        x_c2 = self.L2(x_c1, r2)
        x_c3 = self.L3(x_c2, r2)
        x_c4 = self.L4(x_c3, r1)
        x_c4 = torch.cat([x_c4, x_c1], dim=1)
        x_c5 = self.L5(x_c4, r0)
        x_c5 = torch.cat([x_c5, x_c0], dim=1)
        x_c6 = self.L6(x_c5, nx)
        x_c6 = torch.cat([x_c6, x_fc0], dim=1)
        x_c6 = x_c6.permute(0, 2, 1).contiguous()
        return self.fc2(F.gelu(self.fc1(x_c6))).squeeze(-1)


def count_parameters(model):
    """Standard PyTorch parameter count (one complex entry counts as one)."""
    return sum(parameter.numel() for parameter in model.parameters())


def count_real_scalars(model):
    """Number of stored real scalars (a complex entry contains two reals)."""
    return sum(
        parameter.numel() * (2 if parameter.is_complex() else 1)
        for parameter in model.parameters()
    )


set_seed(CONFIG["BASE_SEED"])
parameter_check_model = UNO1D_H10(x_grid, CONFIG)
UNO1D_H10_PARAMETERS = count_parameters(parameter_check_model)
UNO1D_H10_REAL_SCALARS = count_real_scalars(parameter_check_model)
del parameter_check_model

assert UNO1D_H10_PARAMETERS == 2_830_001, UNO1D_H10_PARAMETERS
assert UNO1D_H10_REAL_SCALARS == 5_557_937, UNO1D_H10_REAL_SCALARS
print("UNO1D-H10 trainable entries :", f"{UNO1D_H10_PARAMETERS:,}")
print("UNO1D-H10 stored real scalars:", f"{UNO1D_H10_REAL_SCALARS:,}")
print("AWFNO reference parameters :", f"{128165:,}")

## Training, recursive validation, checkpointing, and timing

In [ ]:
def state_dict_to_cpu(model):
    return {
        key: value.detach().cpu().clone()
        for key, value in model.state_dict().items()
    }


def optimizer_to_device(optimizer, device):
    for state in optimizer.state.values():
        for key, value in state.items():
            if torch.is_tensor(value):
                state[key] = value.to(device)


def make_scheduler(optimizer):
    return torch.optim.lr_scheduler.StepLR(
        optimizer,
        step_size=CONFIG["SCHEDULER_STEP"],
        gamma=CONFIG["SCHEDULER_GAMMA"],
    )


def train_one_epoch(model, loader, optimizer):
    model.train()
    relative_sum = 0.0
    physical_sum = 0.0
    count = 0

    for history, future in loader:
        history = history.to(DEVICE, non_blocking=True)
        future = future.to(DEVICE, non_blocking=True)
        batch = history.shape[0]

        optimizer.zero_grad(set_to_none=True)
        loss = torch.zeros((), device=DEVICE)
        batch_physical = 0.0
        rolling_history = history
        for step in range(future.shape[-1]):
            truth = future[:, :, step]
            prediction = model(rolling_history)
            loss = loss + relative_l2_sum(prediction, truth)
            with torch.no_grad():
                batch_physical += float(
                    physical_field_l2_torch(prediction, truth).sum().item()
                )
            rolling_history = torch.cat(
                [rolling_history[:, :, 1:], prediction.unsqueeze(-1)], dim=-1
            )

        if not torch.isfinite(loss):
            raise FloatingPointError("UNO1D-H10 produced a non-finite loss.")
        loss.backward()
        optimizer.step()

        relative_sum += float(loss.item())
        physical_sum += batch_physical
        count += batch * future.shape[-1]

    return relative_sum / count, physical_sum / count


@torch.no_grad()
def evaluate_windows(model, loader):
    model.eval()
    relative_sum = 0.0
    physical_sum = 0.0
    count = 0
    for history, future in loader:
        rolling_history = history.to(DEVICE, non_blocking=True)
        future = future.to(DEVICE, non_blocking=True)
        batch = history.shape[0]
        for step in range(future.shape[-1]):
            truth = future[:, :, step]
            prediction = model(rolling_history)
            relative_sum += float(relative_l2_sum(prediction, truth).item())
            physical_sum += float(
                physical_field_l2_torch(prediction, truth).sum().item()
            )
            rolling_history = torch.cat(
                [rolling_history[:, :, 1:], prediction.unsqueeze(-1)], dim=-1
            )
        count += batch * future.shape[-1]
    return relative_sum / count, physical_sum / count


@torch.no_grad()
def recursive_rollout(model, initial_history, nt):
    """Use 10 exact states, then only recursively predicted states."""
    model.eval()
    history = torch.as_tensor(
        initial_history, dtype=torch.float32, device=DEVICE
    )
    batch, nx, history_length = history.shape
    if history_length != CONFIG["HISTORY_LENGTH"]:
        raise ValueError(history.shape)
    rollout = torch.empty(batch, nx, nt, dtype=torch.float32, device=DEVICE)
    rollout[:, :, :history_length] = history

    for step in range(history_length, nt):
        prediction = model(history)
        rollout[:, :, step] = prediction
        history = torch.cat(
            [history[:, :, 1:], prediction.unsqueeze(-1)], dim=-1
        )
    return rollout.cpu().numpy()


@torch.no_grad()
def validation_rollout_error(model, validation):
    prediction = recursive_rollout(
        model,
        validation[:, :, : CONFIG["HISTORY_LENGTH"]],
        validation.shape[2],
    )
    value = rollout_metrics(prediction, validation)["full_mean"]
    del prediction
    gc.collect()
    return value


def fit_or_load_uno1d_h10(n_train):
    run_seed = CONFIG["BASE_SEED"] + int(n_train)
    run_directory = MODEL_DIR / f"N{n_train}"
    run_directory.mkdir(parents=True, exist_ok=True)
    done_path = run_directory / "uno1d_h10_done.pt"
    best_path = run_directory / "uno1d_h10_best.pt"
    latest_path = run_directory / "uno1d_h10_latest.pt"

    if CONFIG["FORCE_RETRAIN"]:
        for stale_path in (done_path, best_path, latest_path):
            if stale_path.exists():
                stale_path.unlink()

    model = UNO1D_H10(x_grid, CONFIG).to(DEVICE)
    assert count_parameters(model) == UNO1D_H10_PARAMETERS

    if done_path.exists() and not CONFIG["FORCE_RETRAIN"]:
        print(f"Loading completed UNO1D-H10 checkpoint: {done_path}")
        checkpoint = torch.load(done_path, map_location=DEVICE)
        model.load_state_dict(checkpoint["model_state"])
        return {
            "model": model,
            "history": checkpoint["history"],
            "best_epoch": int(checkpoint["best_epoch"]),
            "best_val_full": float(checkpoint["best_val_full"]),
            "train_time_sec": float(checkpoint["train_time_sec"]),
            "checkpoint": str(done_path),
        }

    set_seed(run_seed)
    model = UNO1D_H10(x_grid, CONFIG).to(DEVICE)
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=CONFIG["LR"],
        weight_decay=CONFIG["WEIGHT_DECAY"],
    )
    scheduler = make_scheduler(optimizer)

    train_dataset = RolloutWindowDataset(
        U_train_pool[:n_train],
        CONFIG["HISTORY_LENGTH"],
        CONFIG["TRAIN_ROLLOUT_STEPS"],
        CONFIG["TRAIN_WINDOW_STRIDE"],
    )
    val_dataset = RolloutWindowDataset(
        U_val,
        CONFIG["HISTORY_LENGTH"],
        CONFIG["TRAIN_ROLLOUT_STEPS"],
        CONFIG["TRAIN_WINDOW_STRIDE"],
    )
    shuffle_generator = torch.Generator()
    shuffle_generator.manual_seed(run_seed + 100000)
    train_loader = DataLoader(
        train_dataset,
        batch_size=CONFIG["BATCH_SIZE"],
        shuffle=True,
        generator=shuffle_generator,
        num_workers=CONFIG["NUM_WORKERS"],
        pin_memory=CONFIG["PIN_MEMORY"] and DEVICE.type == "cuda",
        drop_last=False,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=CONFIG["EVAL_BATCH_SIZE"],
        shuffle=False,
        num_workers=CONFIG["NUM_WORKERS"],
        pin_memory=CONFIG["PIN_MEMORY"] and DEVICE.type == "cuda",
        drop_last=False,
    )

    history_log = {
        "epoch": [],
        "lr": [],
        "train_relative_l2": [],
        "train_physical_l2": [],
        "val_relative_l2": [],
        "val_physical_l2": [],
        "val_recursive_full_l2": [],
    }
    best_epoch = -1
    best_val_full = float("inf")
    start_epoch = 1
    elapsed_before = 0.0

    if latest_path.exists() and not CONFIG["FORCE_RETRAIN"]:
        latest = torch.load(latest_path, map_location=DEVICE)
        model.load_state_dict(latest["model_state"])
        optimizer.load_state_dict(latest["optimizer_state"])
        optimizer_to_device(optimizer, DEVICE)
        scheduler.load_state_dict(latest["scheduler_state"])
        history_log = latest["history"]
        best_epoch = int(latest["best_epoch"])
        best_val_full = float(latest["best_val_full"])
        start_epoch = int(latest["epoch"]) + 1
        elapsed_before = float(latest.get("train_time_sec", 0.0))
        state = latest.get("shuffle_generator_state")
        if state is not None:
            # torch.load(..., map_location=DEVICE) can move this CPU-generator
            # state onto CUDA. CPU generators require a CPU uint8 tensor.
            if not torch.is_tensor(state):
                state = torch.as_tensor(state, dtype=torch.uint8)
            state = state.detach().cpu().to(dtype=torch.uint8).contiguous()
            shuffle_generator.set_state(state)
        print(f"Resuming N={n_train} at epoch {start_epoch}.")

    print("\n" + "=" * 92)
    print(f"UNO1D-H10 | N_train={n_train}")
    print("=" * 92)
    print("exact input states      :", CONFIG["HISTORY_LENGTH"])
    print("training rollout steps  :", CONFIG["TRAIN_ROLLOUT_STEPS"])
    print("training segments       :", len(train_dataset))
    print("parameters              :", f"{count_parameters(model):,}")
    print("stored real scalars     :", f"{count_real_scalars(model):,}")
    print("batch size              :", CONFIG["BATCH_SIZE"])
    print("test used for selection : no")

    segment_start = time.perf_counter()
    for epoch in range(start_epoch, CONFIG["EPOCHS"] + 1):
        train_rel, train_physical = train_one_epoch(
            model, train_loader, optimizer
        )
        val_rel, val_physical = evaluate_windows(model, val_loader)
        run_rollout = (
            epoch % CONFIG["VAL_ROLLOUT_INTERVAL"] == 0
            or epoch == CONFIG["EPOCHS"]
        )
        val_full = (
            validation_rollout_error(model, U_val)
            if run_rollout else float("nan")
        )

        current_lr = float(optimizer.param_groups[0]["lr"])
        history_log["epoch"].append(epoch)
        history_log["lr"].append(current_lr)
        history_log["train_relative_l2"].append(train_rel)
        history_log["train_physical_l2"].append(train_physical)
        history_log["val_relative_l2"].append(val_rel)
        history_log["val_physical_l2"].append(val_physical)
        history_log["val_recursive_full_l2"].append(val_full)

        if run_rollout and np.isfinite(val_full) and val_full < best_val_full:
            best_val_full = float(val_full)
            best_epoch = int(epoch)
            torch.save(
                {
                    "model_state": state_dict_to_cpu(model),
                    "epoch": best_epoch,
                    "best_val_full": best_val_full,
                    "config": copy.deepcopy(CONFIG),
                },
                best_path,
            )

        scheduler.step()
        if epoch == 1 or epoch % 10 == 0:
            print(
                f"[UNO1D-H10] {epoch:04d}/{CONFIG['EPOCHS']} | "
                f"train rel {train_rel:.6e} | val rel {val_rel:.6e} | "
                f"train physical {train_physical:.6e} | "
                f"val physical {val_physical:.6e} | "
                f"val recursive full {val_full:.6e} | lr {current_lr:.3e}"
            )

        if epoch % 10 == 0 or epoch == CONFIG["EPOCHS"]:
            elapsed = elapsed_before + (time.perf_counter() - segment_start)
            torch.save(
                {
                    "n_train": int(n_train),
                    "epoch": int(epoch),
                    "model_state": state_dict_to_cpu(model),
                    "optimizer_state": optimizer.state_dict(),
                    "scheduler_state": scheduler.state_dict(),
                    "history": history_log,
                    "best_epoch": int(best_epoch),
                    "best_val_full": float(best_val_full),
                    "train_time_sec": float(elapsed),
                    "shuffle_generator_state": shuffle_generator.get_state(),
                    "config": copy.deepcopy(CONFIG),
                },
                latest_path,
            )

    total_train_time = elapsed_before + (time.perf_counter() - segment_start)
    if not best_path.exists():
        raise RuntimeError(f"No finite checkpoint was produced for N={n_train}.")
    best = torch.load(best_path, map_location=DEVICE)
    model.load_state_dict(best["model_state"])

    done_bundle = {
        "model_state": state_dict_to_cpu(model),
        "history": history_log,
        "best_epoch": int(best["epoch"]),
        "best_val_full": float(best["best_val_full"]),
        "train_time_sec": float(total_train_time),
        "parameters": int(count_parameters(model)),
        "stored_real_scalars": int(count_real_scalars(model)),
        "config": copy.deepcopy(CONFIG),
    }
    torch.save(done_bundle, done_path)
    return {
        "model": model,
        "history": history_log,
        "best_epoch": int(best["epoch"]),
        "best_val_full": float(best["best_val_full"]),
        "train_time_sec": float(total_train_time),
        "checkpoint": str(done_path),
    }


def timed_test_rollout(model):
    initial_history = U_test[:, :, : CONFIG["HISTORY_LENGTH"]]
    with torch.no_grad():
        warm_history = torch.as_tensor(
            initial_history, dtype=torch.float32, device=DEVICE
        )
        for _ in range(3):
            warm_prediction = model(warm_history)
            warm_history = torch.cat(
                [warm_history[:, :, 1:], warm_prediction.unsqueeze(-1)], dim=-1
            )
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()

    timings = []
    prediction = None
    for _ in range(CONFIG["INFERENCE_REPEATS"]):
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        start = time.perf_counter()
        prediction = recursive_rollout(model, initial_history, NT)
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        timings.append(time.perf_counter() - start)
    return prediction, float(np.median(timings)), timings

## Plots and complete sample-size sweep

In [ ]:
def plot_history(history, n_train):
    if not CONFIG["SAVE_PLOTS"]:
        return
    epochs = np.asarray(history["epoch"])
    figure, axes = plt.subplots(1, 2, figsize=(12, 4.2))
    axes[0].semilogy(epochs, history["train_relative_l2"], label="train")
    axes[0].semilogy(epochs, history["val_relative_l2"], label="validation")
    axes[0].set_xlabel("epoch")
    axes[0].set_ylabel("mean relative one-step L2 across unrolled training")
    axes[0].grid(True, alpha=0.3)
    axes[0].legend()

    val_full = np.asarray(history["val_recursive_full_l2"], dtype=float)
    finite = np.isfinite(val_full)
    axes[1].semilogy(
        epochs[finite], val_full[finite], marker="o",
        label="validation full-rollout physical L2",
    )
    axes[1].set_xlabel("epoch")
    axes[1].set_ylabel("absolute trajectory L2")
    axes[1].grid(True, alpha=0.3)
    axes[1].legend()
    figure.suptitle(f"UNO1D-H10 training diagnostics, N={n_train}")
    figure.tight_layout()
    figure.savefig(
        PLOT_DIR / f"N{n_train}_uno1d_h10_training.png", dpi=160
    )
    plt.close(figure)


def plot_l2_time(metrics, n_train):
    if not CONFIG["SAVE_PLOTS"]:
        return
    figure, axis = plt.subplots(figsize=(7.5, 4.5))
    axis.plot(t_grid, metrics["mean_t"], label="mean", linewidth=2)
    axis.plot(t_grid, metrics["median_t"], "--", label="median", linewidth=2)
    axis.fill_between(
        t_grid,
        np.maximum(metrics["mean_t"] - metrics["std_t"], 0.0),
        metrics["mean_t"] + metrics["std_t"],
        alpha=0.18,
        label="mean ± std",
    )
    axis.set_xlabel("snapshot time")
    axis.set_ylabel("absolute physical L2")
    axis.set_title(f"UNO1D-H10 recursive rollout, N={n_train}")
    axis.grid(True, alpha=0.3)
    axis.legend()
    figure.tight_layout()
    path = PLOT_DIR / f"N{n_train}_uno1d_h10_l2_per_time.png"
    figure.savefig(path, dpi=160)
    plt.close(figure)

def forecast_only_full_l2(prediction, truth):
    start = CONFIG["HISTORY_LENGTH"] - 1
    l2_t = l2_per_timestep_np(prediction, truth)[:, start:]
    weights = time_weights(l2_t.shape[1])[None, :]
    values = np.sqrt(
        np.sum(weights * l2_t * l2_t, axis=1) * DT_SNAPSHOT + 1.0e-30
    )
    return {
        "forecast_full_mean": float(np.mean(values)),
        "forecast_full_std": float(np.std(values)),
        "forecast_full_median": float(np.median(values)),
    }


all_results = []
for n_train in CONFIG["RUN_SAMPLE_SIZES"]:
    if n_train not in CONFIG["SAMPLE_SIZES"]:
        raise ValueError(f"Unexpected sample size {n_train}.")

    fitted = fit_or_load_uno1d_h10(n_train)
    prediction, inference_median, inference_all = timed_test_rollout(fitted["model"])
    metrics = rollout_metrics(prediction, U_test)
    forecast_metrics = forecast_only_full_l2(prediction, U_test)

    result = {
        "n_train": int(n_train),
        "model": "UNO1D-H10-W32-7BLOCK",
        "parameters": int(count_parameters(fitted["model"])),
        "stored_real_scalars": int(count_real_scalars(fitted["model"])),
        "best_epoch": int(fitted["best_epoch"]),
        "validation_full_l2": float(fitted["best_val_full"]),
        "full_mean": metrics["full_mean"],
        "forecast_full_mean": forecast_metrics["forecast_full_mean"],
        "forecast_full_std": forecast_metrics["forecast_full_std"],
        "forecast_full_median": forecast_metrics["forecast_full_median"],
        "full_std": metrics["full_std"],
        "full_median": metrics["full_median"],
        "final_mean": metrics["final_mean"],
        "final_std": metrics["final_std"],
        "final_median": metrics["final_median"],
        "train_time_sec": float(fitted["train_time_sec"]),
        "inference_time_median_sec": inference_median,
        "inference_time_all_sec": inference_all,
        "checkpoint": fitted["checkpoint"],
    }
    all_results.append(result)

    np.savez_compressed(
        RESULT_DIR / f"N{n_train}_uno1d_h10_metrics.npz",
        mean_t=metrics["mean_t"],
        median_t=metrics["median_t"],
        std_t=metrics["std_t"],
        all_full=metrics["all_full"],
        all_final=metrics["all_final"],
        inference_times=np.asarray(inference_all),
    )
    if CONFIG["SAVE_ROLLOUTS"]:
        np.save(
            RESULT_DIR / f"N{n_train}_uno1d_h10_rollout.npy",
            prediction.astype(np.float32),
        )

    plot_history(fitted["history"], n_train)
    plot_l2_time(metrics, n_train)

    print("\nUNO1D-H10 test result")
    print("-------------------")
    print(f"N_train          : {n_train}")
    print(f"parameters       : {result['parameters']:,}")
    print(f"real scalars     : {result['stored_real_scalars']:,}")
    print(f"full mean/std    : {result['full_mean']:.6e} / {result['full_std']:.6e}")
    print(f"forecast-only L2 : {result['forecast_full_mean']:.6e}")
    print(f"full median      : {result['full_median']:.6e}")
    print(f"final mean/std   : {result['final_mean']:.6e} / {result['final_std']:.6e}")
    print(f"final median     : {result['final_median']:.6e}")
    print(f"train time       : {result['train_time_sec']:.2f} s")
    print(f"inference median : {result['inference_time_median_sec']:.4f} s")

    del fitted, prediction, metrics, forecast_metrics
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

In [ ]:
summary = pd.DataFrame(all_results).sort_values("n_train")
summary_csv = RESULT_DIR / "uno1d_h10_sample_size_summary.csv"
summary_json = RESULT_DIR / "uno1d_h10_sample_size_summary.json"
summary.to_csv(summary_csv, index=False)
with open(summary_json, "w") as stream:
    json.dump(json_ready(all_results), stream, indent=2)

print("\n" + "=" * 104)
print("UNO1D-H10 SAMPLE-SIZE SWEEP: ABSOLUTE PHYSICAL L2")
print("=" * 104)
for row in all_results:
    print(
        f"N={row['n_train']:4d} | "
        f"full mean {row['full_mean']:.6e} | "
        f"std {row['full_std']:.6e} | "
        f"forecast-only {row['forecast_full_mean']:.6e} | "
        f"median {row['full_median']:.6e} | "
        f"final mean {row['final_mean']:.6e} | "
        f"final median {row['final_median']:.6e} | "
        f"params {row['parameters']:,} | "
        f"real scalars {row['stored_real_scalars']:,} | "
        f"train {row['train_time_sec']:.2f}s | "
        f"inference {row['inference_time_median_sec']:.4f}s"
    )
print("=" * 104)
print("Saved CSV :", summary_csv)
print("Saved JSON:", summary_json)

if CONFIG["SAVE_PLOTS"] and len(summary) > 1:
    figure, axes = plt.subplots(1, 2, figsize=(11, 4.3))
    axes[0].loglog(
        summary["n_train"], summary["full_mean"],
        marker="o", linewidth=2, label="UNO1D-H10",
    )
    axes[0].set_xlabel("training trajectories")
    axes[0].set_ylabel("mean full-trajectory L2")
    axes[0].grid(True, which="both", alpha=0.3)
    axes[0].legend()

    axes[1].loglog(
        summary["n_train"], summary["final_mean"],
        marker="o", linewidth=2, label="UNO1D-H10",
    )
    axes[1].set_xlabel("training trajectories")
    axes[1].set_ylabel("mean final-time L2")
    axes[1].grid(True, which="both", alpha=0.3)
    axes[1].legend()

    figure.tight_layout()
    figure.savefig(PLOT_DIR / "uno1d_h10_sample_size_scaling.png", dpi=170)
    plt.show()

## Optional: release the Colab runtime

Run the next two lines only after confirming that the checkpoints and result
tables were written to Google Drive.

In [ ]:
from google.colab import runtime
runtime.unassign()